In [1]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
ID_COLUMN = "front_id"
TARGET_COLUMN = "target_value"
DATE_COLUMN = "decision_day"
START_DATE = pd.Timestamp("2024-02-01")
HOLDOUT_MONTHS = 2
N_SPLITS = 5
RANDOM_STATE = 42

MISSINGNESS_BLOCKS = {
    "corp_block_isna": [
        "corp_credit_products",
        "corp_list",
        "count_all_corp_dashboard_events",
        "p75_time_spent_minutes",
    ],
    "deb_cred_loan_isna": ["cnt_deb_loan_90", "cnt_cred_loan_90"],
    "app_term_db_group_isna": ["app_term_mean_360", "db_group_last"],
}

SINGLE_MISSING_COLUMNS = [
    "sum_deb_ul_90",
    "sum_deb_ul_30",
    "cnt_deb_ul_ip_90",
    "cnt_deb_ul_ip_30",
    "balance_rur_amt_30_min",
    "loan_rev_max_start_non_fin",
    "loan_rev_min_start_fin",
    "overdraft_app_term_max_360",
    "days_from_authperson_registration",
    "fl_hdb_bki_total_active_products",
    "sum_deb_investment_90",
    "fl_adminarea",
]

CATEGORICAL_COLUMNS = ["db_group_last", "fl_adminarea"]

RARE_COLUMNS = [
    "overdraft_app_term_max_360",
    "loan_rev_max_start_non_fin",
    "loan_rev_min_start_fin",
    "sum_deb_investment_90",
]

ALL_MISSING_COLUMNS = [
    column for columns in MISSINGNESS_BLOCKS.values() for column in columns
] + SINGLE_MISSING_COLUMNS

XGB_PARAMS = {
    "max_depth": 7,
    "learning_rate": 0.035,
    "min_child_weight": 24,
    "subsample": 0.66,
    "colsample_bytree": 0.79,
    "reg_lambda": 1.27,
    "reg_alpha": 0.053,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "enable_categorical": True,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

CATBOOST_PARAMS = {
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 3.0,
    "eval_metric": "AUC",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
}

In [3]:
def build_features(data):
    features = data.copy()

    parsed_date = pd.to_datetime(features[DATE_COLUMN])
    features["month"] = parsed_date.dt.month.astype("int16")
    features["dayofweek"] = parsed_date.dt.dayofweek.astype("int16")
    features["day"] = parsed_date.dt.day.astype("int16")
    features["days_since_start"] = (parsed_date - START_DATE).dt.days.astype("int32")

    features["n_missing"] = features[ALL_MISSING_COLUMNS].isna().sum(axis=1).astype("int16")
    features["n_present_rare"] = features[RARE_COLUMNS].notna().sum(axis=1).astype("int8")

    for indicator_name, block_columns in MISSINGNESS_BLOCKS.items():
        features[indicator_name] = features[block_columns[0]].isna().astype("int8")
    for column in SINGLE_MISSING_COLUMNS:
        features[f"{column}_isna"] = features[column].isna().astype("int8")

    features["loan_amount_vs_overdraft_limit"] = features["loan_amount_last"] - features["overdraft_limit_max"]
    features["offered_rate_spread"] = features["offered_rate"] - features["cb_rate"]
    features["overdraft_limit_range"] = features["overdraft_limit_max"] - features["overdraft_limit_min"]
    features["debit_ip_trend_30_90"] = features["cnt_deb_ul_ip_30"] - features["cnt_deb_ul_ip_90"]
    features["debit_sum_trend_30_90"] = features["sum_deb_ul_30"] - features["sum_deb_ul_90"]
    features["debit_vs_credit_loans"] = features["cnt_deb_loan_90"] - features["cnt_cred_loan_90"]
    features["offered_rate_clipped"] = features["offered_rate"].clip(-5, 5)

    for column in CATEGORICAL_COLUMNS:
        features[column] = features[column].fillna("missing").astype("category")

    columns_to_drop = [ID_COLUMN, DATE_COLUMN]
    if TARGET_COLUMN in features.columns:
        columns_to_drop.append(TARGET_COLUMN)
    return features.drop(columns=columns_to_drop)

In [4]:
def align_categories(train_features, test_features):
    for column in CATEGORICAL_COLUMNS:
        categories = train_features[column].cat.categories.union(test_features[column].cat.categories)
        train_features[column] = train_features[column].cat.set_categories(categories)
        test_features[column] = test_features[column].cat.set_categories(categories)
    return train_features, test_features[train_features.columns]


def with_string_categories(features):
    string_features = features.copy()
    for column in CATEGORICAL_COLUMNS:
        string_features[column] = string_features[column].astype(str)
    return string_features


def rank_average(*score_arrays):
    return sum(rankdata(scores) for scores in score_arrays) / len(score_arrays)


def fit_xgboost(train_features, train_target, n_estimators, validation=None):
    model = XGBClassifier(n_estimators=n_estimators, **XGB_PARAMS)
    if validation is None:
        model.fit(train_features, train_target, verbose=False)
    else:
        model.set_params(early_stopping_rounds=150)
        model.fit(train_features, train_target, eval_set=[validation], verbose=False)
    return model


def fit_catboost(train_features, train_target, iterations, validation=None):
    model = CatBoostClassifier(iterations=iterations, **CATBOOST_PARAMS)
    if validation is None:
        model.fit(train_features, train_target, cat_features=CATEGORICAL_COLUMNS)
    else:
        model.set_params(early_stopping_rounds=150)
        model.fit(train_features, train_target, cat_features=CATEGORICAL_COLUMNS, eval_set=validation)
    return model

In [5]:
train = pd.read_csv("train_apps.csv")
test = pd.read_csv("test_apps.csv")

target = train[TARGET_COLUMN]
train_dates = pd.to_datetime(train[DATE_COLUMN])

features = build_features(train)
test_features = build_features(test)
features, test_features = align_categories(features, test_features)
string_features = with_string_categories(features)
string_test_features = with_string_categories(test_features)

In [6]:
is_holdout = (train_dates > train_dates.max() - pd.DateOffset(months=HOLDOUT_MONTHS)).values
holdout_validation = (features[is_holdout], target[is_holdout])
string_holdout_validation = (string_features[is_holdout], target[is_holdout])

xgb_holdout = fit_xgboost(features[~is_holdout], target[~is_holdout], 3000, holdout_validation)
catboost_holdout = fit_catboost(string_features[~is_holdout], target[~is_holdout], 3000, string_holdout_validation)

xgb_rounds = xgb_holdout.best_iteration + 1
catboost_rounds = catboost_holdout.best_iteration_ + 1

xgb_holdout_scores = xgb_holdout.predict_proba(features[is_holdout])[:, 1]
catboost_holdout_scores = catboost_holdout.predict_proba(string_features[is_holdout])[:, 1]
holdout_target = target[is_holdout]

print("holdout xgb     ", round(roc_auc_score(holdout_target, xgb_holdout_scores), 5))
print("holdout catboost", round(roc_auc_score(holdout_target, catboost_holdout_scores), 5))
print("holdout ensemble", round(roc_auc_score(holdout_target, rank_average(xgb_holdout_scores, catboost_holdout_scores)), 5))

holdout xgb      0.75155
holdout catboost 0.75333
holdout ensemble 0.75561


In [7]:
folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
xgb_oof = np.zeros(len(features))
catboost_oof = np.zeros(len(features))

for train_index, valid_index in folds.split(features, target):
    xgb_fold = fit_xgboost(features.iloc[train_index], target.iloc[train_index], xgb_rounds)
    xgb_oof[valid_index] = xgb_fold.predict_proba(features.iloc[valid_index])[:, 1]

    catboost_fold = fit_catboost(string_features.iloc[train_index], target.iloc[train_index], catboost_rounds)
    catboost_oof[valid_index] = catboost_fold.predict_proba(string_features.iloc[valid_index])[:, 1]

print("oof xgb     ", round(roc_auc_score(target, xgb_oof), 5))
print("oof catboost", round(roc_auc_score(target, catboost_oof), 5))
print("oof ensemble", round(roc_auc_score(target, rank_average(xgb_oof, catboost_oof)), 5))

xgb_oof_by_id = pd.Series(xgb_oof, index=train[ID_COLUMN].values)
catboost_oof_by_id = pd.Series(catboost_oof, index=train[ID_COLUMN].values)

oof xgb      0.83577
oof catboost 0.83506


oof ensemble 0.83716


In [8]:
xgb_model = fit_xgboost(features, target, int(xgb_rounds * 1.1))
catboost_model = fit_catboost(string_features, target, int(catboost_rounds * 1.1))

xgb_test_scores = pd.Series(xgb_model.predict_proba(test_features)[:, 1], index=test[ID_COLUMN].values)
catboost_test_scores = pd.Series(catboost_model.predict_proba(string_test_features)[:, 1], index=test[ID_COLUMN].values)

xgb_scores_by_id = pd.concat([xgb_test_scores, xgb_oof_by_id])
catboost_scores_by_id = pd.concat([catboost_test_scores, catboost_oof_by_id])
xgb_scores_by_id = xgb_scores_by_id[~xgb_scores_by_id.index.duplicated()]
catboost_scores_by_id = catboost_scores_by_id[~catboost_scores_by_id.index.duplicated()]

submission = pd.read_csv("sample_submission.csv")[[ID_COLUMN]].copy()
xgb_aligned = submission[ID_COLUMN].map(xgb_scores_by_id).fillna(xgb_scores_by_id.median())
catboost_aligned = submission[ID_COLUMN].map(catboost_scores_by_id).fillna(catboost_scores_by_id.median())

ensemble_scores = rank_average(xgb_aligned.values, catboost_aligned.values)
submission[TARGET_COLUMN] = (ensemble_scores - ensemble_scores.min()) / (ensemble_scores.max() - ensemble_scores.min())
submission.to_csv("submission.csv", index=False)
submission.head()

,front_id,target_value
0,238459,0.086834
1,151533,0.305969
2,195027,0.593570
3,195034,0.613549
4,129620,0.607552
